In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import load_model, Model
from datetime import datetime
import subprocess

# ── TPU INIT ───────────────────────────────────────────
resolver = tf.distribute.cluster_resolver.TPUClusterResolver()
tf.config.experimental_connect_to_cluster(resolver)
tf.tpu.experimental.initialize_tpu_system(resolver)
strategy = tf.distribute.TPUStrategy(resolver)

print(f"TPU replicas : {strategy.num_replicas_in_sync}")
print(f"TF version   : {tf.__version__}")

# ── PATHS ──────────────────────────────────────────────
# Use central2 bucket since TPU is in us-central2-b
MODEL_PATH    = 'gs://tala-sentinel2-data/models/proxy_cnn/proxy_cnn_final.keras'
GCS_TFRECORDS = 'gs://tala-sentinel2-data-central2/tfrecords/sentinel2'
OUTPUT_GCS    = 'gs://tala-sentinel2-data/features/dynamic'
OUTPUT_LOCAL  = '/home/tpu-vm/dynamic_features_full.csv'

# Batch size: 32 per replica × 8 replicas = 256
REPLICAS      = strategy.num_replicas_in_sync
BATCH_SIZE    = REPLICAS * 32
AUTOTUNE      = tf.data.AUTOTUNE

print(f"Batch size   : {BATCH_SIZE}")

# Verify GCS access
result = subprocess.run(
    ['gsutil', 'ls', f'{GCS_TFRECORDS}/'],
    capture_output=True, text=True
)
shards = [l for l in result.stdout.strip().split('\n')
          if '.tfrecord' in l]
print(f"Shards found : {len(shards)}")

In [ ]:
# Must load model inside strategy.scope() for TPU
print("Loading feature extractor...")

with strategy.scope():
    full_model = load_model(MODEL_PATH)
    feature_extractor = Model(
        inputs  = full_model.input,
        outputs = full_model.get_layer('feature_vector').output
    )

print(f"Input  : {feature_extractor.input_shape}")
print(f"Output : {feature_extractor.output_shape}")
print("✓ Feature extractor ready")

In [ ]:
feature_spec = {
    'image'     : tf.io.FixedLenFeature([], tf.string),
    'cluster_id': tf.io.FixedLenFeature([], tf.int64),
    'quarter'   : tf.io.FixedLenFeature([], tf.int64),
    'height'    : tf.io.FixedLenFeature([], tf.int64),
    'width'     : tf.io.FixedLenFeature([], tf.int64),
    'channels'  : tf.io.FixedLenFeature([], tf.int64),
}

def parse_for_extraction(example_proto):
    parsed = tf.io.parse_single_example(
        example_proto, feature_spec)
    h   = tf.cast(parsed['height'],   tf.int32)
    w   = tf.cast(parsed['width'],    tf.int32)
    c   = tf.cast(parsed['channels'], tf.int32)
    img = tf.io.decode_raw(parsed['image'], tf.float32)
    img = tf.reshape(img, [h, w, c])

    # Clean NaNs
    img = tf.where(tf.math.is_nan(img),
                   tf.zeros_like(img), img)
    img = tf.clip_by_value(img, 0.0, 1.0)

    # VGG preprocessing (must match training)
    img = img * 255.0
    img = tf.keras.applications.vgg16.preprocess_input(img)

    return img, parsed['cluster_id'], parsed['quarter']

all_shards = tf.io.gfile.glob(
    f'{GCS_TFRECORDS}/*.tfrecord')
print(f"Total shards: {len(all_shards)}")

ds = tf.data.TFRecordDataset(
    all_shards, num_parallel_reads=AUTOTUNE)
ds = ds.map(parse_for_extraction,
            num_parallel_calls=AUTOTUNE)
# drop_remainder=True required for TPU
ds = ds.batch(BATCH_SIZE, drop_remainder=True)
ds = ds.prefetch(AUTOTUNE)

# Sanity check
for imgs, cids, qs in ds.take(1):
    print(f"Batch shape : {imgs.shape}")
    print(f"Cluster IDs : {cids.numpy()[:4]}")
    print(f"Quarters    : {qs.numpy()[:4]}")
print("✓ Dataset ready")

In [ ]:
records   = []
batch_num = 0
start     = datetime.now()

print("Extracting features...")

for imgs, cluster_ids, quarters in ds:
    feats = feature_extractor.predict(imgs, verbose=0)

    for i in range(len(cluster_ids)):
        cid = int(cluster_ids[i].numpy())
        q   = f'Q{int(quarters[i].numpy())}'
        rec = {'ClusterID': cid, 'Quarter': q}
        rec.update({
            f'CNN_{k}': float(feats[i][k])
            for k in range(feats.shape[1])
        })
        records.append(rec)

    batch_num += 1
    if batch_num % 5 == 0:
        elapsed = (datetime.now() - start).seconds
        print(f"  [{datetime.now().strftime('%H:%M:%S')}] "
              f"Batch {batch_num} | "
              f"Records: {len(records)} | "
              f"Elapsed: {elapsed}s")

# Note: drop_remainder=True means the last partial batch
# is dropped. Recover these few records below.
print(f"\nMain extraction done. Records: {len(records)}")
print(f"Total time: "
      f"{(datetime.now()-start).seconds} seconds")

# Save
df = pd.DataFrame(records).sort_values(
    ['ClusterID', 'Quarter']).reset_index(drop=True)
df.to_csv(OUTPUT_LOCAL, index=False)

subprocess.run([
    'gsutil', 'cp', OUTPUT_LOCAL,
    f'{OUTPUT_GCS}/dynamic_features_full.csv'
])

print(f"\n{'='*50}")
print("FEATURE EXTRACTION COMPLETE")
print(f"{'='*50}")
print(f"Records         : {len(df)}")
print(f"Unique clusters : {df['ClusterID'].nunique()}")
print(f"Quarters        : "
      f"{sorted(df['Quarter'].unique())}")
print(f"Feature dims    : "
      f"{df.shape[1] - 2} (should be 4096)")
print(f"Saved to GCS    : "
      f"{OUTPUT_GCS}/dynamic_features_full.csv")

In [ ]:
# Recover the dropped tail records
# Switch to CPU prediction for the small remainder

print("Recovering tail records dropped by TPU batching...")

# Find which cluster/quarter pairs are missing
done_keys = set(zip(
    df['ClusterID'].astype(int),
    df['Quarter'].astype(str)
))

# Rebuild dataset without drop_remainder
ds_tail = tf.data.TFRecordDataset(
    all_shards, num_parallel_reads=AUTOTUNE)
ds_tail = ds_tail.map(parse_for_extraction,
                      num_parallel_calls=AUTOTUNE)
ds_tail = ds_tail.batch(32, drop_remainder=False)
ds_tail = ds_tail.prefetch(AUTOTUNE)

tail_records = []
for imgs, cluster_ids, quarters in ds_tail:
    batch_cids = cluster_ids.numpy()
    batch_qs   = [f'Q{int(q)}' for q in quarters.numpy()]

    # Only process records not already extracted
    new_idx = [
        i for i in range(len(batch_cids))
        if (int(batch_cids[i]), batch_qs[i])
        not in done_keys
    ]

    if not new_idx:
        continue

    imgs_new = tf.gather(imgs, new_idx).numpy()
    # Use CPU prediction for tail
    feats = feature_extractor.predict(
        imgs_new, verbose=0)

    for j, i in enumerate(new_idx):
        cid = int(batch_cids[i])
        q   = batch_qs[i]
        rec = {'ClusterID': cid, 'Quarter': q}
        rec.update({
            f'CNN_{k}': float(feats[j][k])
            for k in range(feats.shape[1])
        })
        tail_records.append(rec)
        done_keys.add((cid, q))

print(f"Tail records recovered: {len(tail_records)}")

if tail_records:
    df_tail  = pd.DataFrame(tail_records)
    df_final = pd.concat(
        [df, df_tail], ignore_index=True
    ).sort_values(
        ['ClusterID', 'Quarter']
    ).reset_index(drop=True)

    df_final.to_csv(OUTPUT_LOCAL, index=False)
    subprocess.run([
        'gsutil', 'cp', OUTPUT_LOCAL,
        f'{OUTPUT_GCS}/dynamic_features_full.csv'
    ])
    print(f"Final record count: {len(df_final)}")
else:
    df_final = df
    print("No tail records needed.")

print(f"\nFinal output:")
print(f"  Records         : {len(df_final)}")
print(f"  Unique clusters : {df_final['ClusterID'].nunique()}")
print(f"  GCS path        : "
      f"{OUTPUT_GCS}/dynamic_features_full.csv")
print(f"\nCopy to Drive:")
print(f"  gsutil cp {OUTPUT_GCS}/dynamic_features_full.csv "
      f"gs://[your-drive-bucket]/")
print(f"\nThen run merge and LSTM on M4.")